# Czech UPOS v2 — stage-oriented Kaggle frontend

This notebook is a thin frontend for the manifest-driven `sanna-tagging` v2 CLI. It contains no training or selection logic. Run **one stage per committed Kaggle version**.

## Required Kaggle procedure

1. Settings → Accelerator = GPU and Internet = On.
2. The notebook is pinned to runtime commit `abb04f9a9f3f0d6c44fab13a0eba8c1ce4613579`; do not change it between stages.
3. Choose `STAGE`: `prepare`, `budget`, `adapt`, `report-draft`, or (only after separate authorization) `finalize-test`.
4. Use **Save Version → Save & Run All (Commit)**. Do not rely on an overnight interactive session: Kaggle removes `/kaggle/working` when it ends, while a committed version permanently attaches that directory as Output.
5. For the next stage, add the previous committed version's Output as a read-only Kaggle input and set `PREVIOUS_RUN_DIR` to its exact `/kaggle/input/.../runs/cs-rerun-v2/<fingerprint>` directory.

Prior output is never modified. Its fingerprint, identity, generic manifest outputs, authorized checkpoint-pruning records, and retained checkpoint hashes are verified before it is copied to the new writable `/kaggle/working/sanna-v2/runs` tree. A mismatch fails closed. Protected raw Czech test gold uses an ephemeral cache outside `/kaggle/working`; only FORM-only test data enters the portable run.

`report-draft` is the normal final stage. `finalize-test` must run only in a separate deliberately configured committed version after the draft and selection lock have been reviewed. It requires `AUTHORIZE_FINAL_TEST = True` and an official test path under `/kaggle/input`; a failed attempt is still consumed by the one-shot marker.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

# Change only these run parameters between committed Kaggle versions.
STAGE = "prepare"
GIT_COMMIT = "abb04f9a9f3f0d6c44fab13a0eba8c1ce4613579"
PREVIOUS_RUN_DIR = ""  # Exact /kaggle/input/.../<run-name>/<fingerprint> directory.
AUTHORIZE_FINAL_TEST = False
OFFICIAL_TEST_PATH = ""  # Required only for an explicitly authorized finalize-test version.

REPO_URL = "https://github.com/nikolask11/Sanna-tagging.git"
WORK_ROOT = Path("/kaggle/working/sanna-v2")
REPO = WORK_ROOT / "repo"
RUNS_ROOT = WORK_ROOT / "runs"
CONFIG = REPO / "configs/cs_kaggle_v2.yaml"

if not re.fullmatch(r"[0-9a-f]{40}", GIT_COMMIT):
    raise ValueError("GIT_COMMIT must be the exact lowercase 40-character runtime commit")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO)], check=True)
subprocess.run(
    ["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", GIT_COMMIT],
    check=True,
)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", GIT_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if head != GIT_COMMIT:
    raise RuntimeError(f"detached checkout mismatch: expected {GIT_COMMIT}, got {head}")

lock = REPO / "requirements-kaggle.lock"
lock_entries = [
    line.strip()
    for line in lock.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
if not lock_entries or any("==" not in entry for entry in lock_entries):
    raise RuntimeError("requirements-kaggle.lock must contain exact pins only")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--disable-pip-version-check",
        "-r",
        str(lock),
    ],
    check=True,
)

# Kaggle preinstalls torchvision/torchaudio built against a newer torch than the lock
# pins, which breaks `import transformers` with "operator torchvision::nms does not
# exist". Neither package is used here, so drop them instead of widening the lock.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"],
    check=False,
)
importlib.invalidate_caches()
for stale in ("torchvision", "torchaudio"):
    if importlib.util.find_spec(stale) is not None:
        raise RuntimeError(f"{stale} is still installed and conflicts with the pinned torch")

os.environ["PYTHONPATH"] = str(REPO / "src")
sys.path.insert(0, str(REPO / "src"))
import torch

if STAGE in {"budget", "adapt"} and not torch.cuda.is_available():
    raise RuntimeError(f"STAGE={STAGE!r} requires a Kaggle GPU accelerator")
print("Detached commit:", head)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not required for this stage")
print("Portable output root:", WORK_ROOT)

In [ ]:
import json

from sanna_tagging.artifacts import load_manifest, sha256_file
from sanna_tagging.runner import ExperimentRunner

runner = ExperimentRunner(config_path=CONFIG, runs_root=RUNS_ROOT, code_commit=GIT_COMMIT)
expected_fingerprint = runner.fingerprint
destination_run = runner.layout.run_dir
print("Expected fingerprint:", expected_fingerprint)
print("Writable run:", destination_run)


def verify_record(record, *, bases, source_run):
    if not isinstance(record, dict):
        raise RuntimeError("malformed file record")
    value = record.get("path")
    if not isinstance(value, str):
        raise RuntimeError("file record path is missing")
    logical = Path(value)
    if logical.is_absolute() or ".." in logical.parts:
        raise RuntimeError(f"unsafe committed path: {value}")
    matches = []
    for base in bases:
        candidate = (base / logical).resolve()
        try:
            candidate.relative_to(source_run)
        except ValueError:
            continue
        if (
            candidate.is_file()
            and candidate.stat().st_size == record.get("bytes")
            and sha256_file(candidate) == record.get("sha256")
        ):
            matches.append(candidate)
    if not matches:
        raise RuntimeError(f"missing or damaged committed file: {value}")
    return matches[0]


def safe_run_path(source_run, value):
    logical = Path(str(value))
    if logical.is_absolute() or ".." in logical.parts:
        raise RuntimeError(f"unsafe run-relative path: {value}")
    path = (source_run / logical).resolve()
    path.relative_to(source_run)
    return path


def verify_previous_run(source_run):
    source_run = source_run.resolve()
    try:
        source_run.relative_to(Path("/kaggle/input").resolve())
    except ValueError as error:
        raise RuntimeError("PREVIOUS_RUN_DIR must be a read-only /kaggle/input path") from error
    if not source_run.is_dir() or source_run.name != expected_fingerprint:
        raise RuntimeError("previous run path/fingerprint mismatch")
    if (source_run / "final").exists():
        raise RuntimeError("a finalized run cannot continue into another stage")
    for path in source_run.rglob("*"):
        if path.is_symlink():
            raise RuntimeError(f"previous output contains a forbidden symlink: {path}")
        if path.suffix == ".conllu":
            raise RuntimeError(f"portable run contains forbidden raw CoNLL-U: {path}")

    identity_path = source_run / "plan/run.identity.json"
    identity = json.loads(identity_path.read_text(encoding="utf-8"))
    if (
        identity.get("run_fingerprint") != expected_fingerprint
        or identity.get("run_name") != runner.config["run_name"]
    ):
        raise RuntimeError("previous run identity does not match this checkout/config/lock")

    manifests = sorted(
        set(source_run.rglob("artifact-manifest.json"))
        | set(source_run.rglob("*.manifest.json"))
    )
    if not manifests:
        raise RuntimeError("previous run contains no commit manifests")

    checkpoint_manifests = []
    generic_manifests = []
    for manifest_path in manifests:
        if manifest_path.name == "checkpoint.manifest.json":
            checkpoint_manifests.append(manifest_path)
        else:
            generic_manifests.append(manifest_path)

    authorized_pruned = set()
    for manifest_path in generic_manifests:
        value = load_manifest(manifest_path)
        if value.get("run_fingerprint") != expected_fingerprint:
            raise RuntimeError(f"manifest fingerprint mismatch: {manifest_path}")
        outputs = value.get("outputs")
        if not isinstance(outputs, dict) or not outputs:
            raise RuntimeError(f"generic manifest has no outputs: {manifest_path}")
        verified_outputs = {
            name: verify_record(
                record,
                bases=(source_run, manifest_path.parent),
                source_run=source_run,
            )
            for name, record in outputs.items()
        }
        if manifest_path.name == "pruning.manifest.json":
            plan_path = verified_outputs.get("plan")
            if plan_path is None:
                raise RuntimeError(f"pruning manifest has no verified plan: {manifest_path}")
            plan = json.loads(plan_path.read_text(encoding="utf-8"))
            if plan.get("run_fingerprint") != expected_fingerprint:
                raise RuntimeError(f"pruning plan fingerprint mismatch: {plan_path}")
            for entry in plan.get("entries", []):
                if not isinstance(entry, dict):
                    raise RuntimeError(f"malformed pruning entry: {plan_path}")
                if entry.get("retained") is False:
                    checkpoint_manifest = safe_run_path(
                        source_run, entry.get("checkpoint_manifest")
                    )
                    if (
                        not checkpoint_manifest.is_file()
                        or sha256_file(checkpoint_manifest)
                        != entry.get("checkpoint_manifest_sha256")
                    ):
                        raise RuntimeError(
                            f"pruned checkpoint seal mismatch: {checkpoint_manifest}"
                        )
                    checkpoint_root = safe_run_path(
                        source_run, entry.get("checkpoint_root")
                    )
                    if checkpoint_root.exists():
                        raise RuntimeError(
                            f"authorized pruned checkpoint still exists: {checkpoint_root}"
                        )
                    authorized_pruned.add(checkpoint_manifest)

    for manifest_path in checkpoint_manifests:
        if manifest_path.resolve() in authorized_pruned:
            continue
        value = json.loads(manifest_path.read_text(encoding="utf-8"))
        if value.get("run_fingerprint") != expected_fingerprint:
            raise RuntimeError(f"checkpoint fingerprint mismatch: {manifest_path}")
        records = value.get("files")
        if not isinstance(records, dict) or not records:
            raise RuntimeError(f"checkpoint manifest has no files: {manifest_path}")
        for record in records.values():
            verify_record(record, bases=(manifest_path.parent,), source_run=source_run)
        checkpoint_root = Path(str(value.get("checkpoint_root", "")))
        if checkpoint_root.is_absolute() or ".." in checkpoint_root.parts:
            raise RuntimeError(f"unsafe checkpoint root: {manifest_path}")
        if not (manifest_path.parent / checkpoint_root).is_dir():
            raise RuntimeError(f"checkpoint root is missing: {manifest_path}")
    return source_run, len(manifests)


if PREVIOUS_RUN_DIR:
    source_run, manifest_count = verify_previous_run(Path(PREVIOUS_RUN_DIR))
    if destination_run.exists():
        raise RuntimeError(f"refusing to merge into existing writable run: {destination_run}")
    destination_run.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_run, destination_run)
    print(f"Verified {manifest_count} manifests before copying read-only resume state")
elif STAGE != "prepare":
    raise RuntimeError(f"STAGE={STAGE!r} requires PREVIOUS_RUN_DIR from a committed prior version")

In [ ]:
ALLOWED_STAGES = {"prepare", "budget", "adapt", "report-draft", "finalize-test"}
if STAGE not in ALLOWED_STAGES:
    raise ValueError(f"STAGE must be one of {sorted(ALLOWED_STAGES)}")

command = [
    sys.executable,
    "-m",
    "sanna_tagging.cli",
    "--config",
    str(CONFIG),
    "--runs-root",
    str(RUNS_ROOT),
    STAGE,
]
if STAGE == "finalize-test":
    if AUTHORIZE_FINAL_TEST is not True:
        raise RuntimeError("finalize-test requires AUTHORIZE_FINAL_TEST = True in a separate version")
    official = Path(OFFICIAL_TEST_PATH).resolve()
    try:
        official.relative_to(Path("/kaggle/input").resolve())
    except ValueError as error:
        raise RuntimeError("OFFICIAL_TEST_PATH must be a read-only /kaggle/input file") from error
    if not official.is_file():
        raise FileNotFoundError(official)
    selection_lock = destination_run / "report/selection.lock.json"
    command.extend(
        ["--selection-lock", str(selection_lock), "--official-test", str(official)]
    )

print("Running one v2 CLI stage:", " ".join(command))
subprocess.run(command, check=True, cwd=REPO, env=os.environ.copy())

handoff = destination_run / STAGE / "handoff.json"
if STAGE == "prepare":
    handoff = destination_run / "prepared/dataset_manifest.json"
elif STAGE == "report-draft":
    handoff = destination_run / "report/selection.lock.json"
elif STAGE == "finalize-test":
    handoff = destination_run / "final/REPORT_FINAL.md"
print("Committed stage artifact:", handoff)
print("Kaggle Output to attach to the next version:", WORK_ROOT)
if STAGE == "report-draft":
    print((destination_run / "report/report.draft.md").read_text(encoding="utf-8"))
    print("STOP: review the draft and frozen lock; official test is not evaluated.")
elif STAGE == "finalize-test":
    print(handoff.read_text(encoding="utf-8"))